# Metrics

The two error measures every comparison uses. Both take the actual values and the
forecast values for the same days.

In [ ]:
#| default_exp metrics

In [ ]:
#| export
from collections.abc import Sequence
from math import fsum, isfinite

Shared input check: same length, not empty, no NaN or infinity.

In [ ]:
#| export
def _check_pair(actual: Sequence[float], predicted: Sequence[float]) -> None:
    if not actual:
        raise ValueError("actual must be nonempty")
    if len(actual) != len(predicted):
        raise ValueError("actual and predicted must have equal lengths")
    if not all(isfinite(value) for value in (*actual, *predicted)):
        raise ValueError("actual and predicted must contain only finite values")

MAE: how big the misses are on average, ignoring direction.

In [ ]:
#| export
def mae(actual: Sequence[float], predicted: Sequence[float]) -> float:
    """Mean absolute error: the average size of the misses."""
    _check_pair(actual, predicted)
    return fsum(
        abs(observed - forecast) for observed, forecast in zip(actual, predicted)
    ) / len(actual)

Signed bias: forecast minus actual, averaged. Negative means the forecast is too low.

In [ ]:
#| export
def signed_bias(actual: Sequence[float], predicted: Sequence[float]) -> float:
    """Mean of forecast minus actual; negative means underprediction."""
    _check_pair(actual, predicted)
    return fsum(
        forecast - observed for observed, forecast in zip(actual, predicted)
    ) / len(actual)

## Examples

In [ ]:
print("MAE:", mae([1.0, 5.0], [3.0, 2.0]))
print("bias (too low):", signed_bias([10.0, 10.0], [8.0, 9.0]))
print("bias (too high):", signed_bias([10.0], [12.0]))
assert mae([1.0, 5.0], [3.0, 2.0]) == 2.5
assert signed_bias([10.0, 10.0], [8.0, 9.0]) == -1.5
assert signed_bias([10.0], [12.0]) == 2.0